# Minimap Dot Detection — De-risk Spike (Phase 0)

**目的**: 10 個のミニマップドット (味方 5・敵 5) を YOLO 再学習で安定識別できるかを 1 日で判定する。

**判定基準**:
- `enemy_dot` の mAP@50 ≥ 0.7 → 本学習で進める（Phase 4 のスコープを「10 人個別トラッキング」で確定）
- それ未満 → HSV 色バケット + ハンガリアン法フォールバックを Phase 4 のベースラインに採用

**追跡 Issue**: tsukasakaneko/tauriAICoaching#9 / Phase 4: #13

**前提 (このリポジトリの実装に合わせた値)**:
- 録画は `~/.local/share/valorant-ai-coaching/match_*.mp4`（`screenRecorder.js`）
- 録画解像度は **1280×720**（`screenRecorder.js` の `-s 1280x720` / `scale=1280:720`）
- VALORANT のミニマップは画面左上。`MINIMAP_BBOX` は §0.5 のキャリブレーションで一度合わせる

**使い方**: 上から順に実行。GPU 学習 (§4-5) は Colab 推奨。HSV フォールバック (§6) は録画なしでも合成テストで動作確認できる。

## 1. セットアップ

In [ ]:
# 依存 (Colab/未インストール環境のみ)
# !pip install ultralytics opencv-python-headless pillow numpy scipy pyyaml

import os
import subprocess
import tempfile
from collections import Counter
from pathlib import Path

import numpy as np

# --- パス設定 ---
# 録画の出力先 (screenRecorder.js と一致)。環境変数 RECORDINGS_DIR で上書き可。
RECORDINGS_DIR = Path(
    os.environ.get('RECORDINGS_DIR', Path.home() / '.local' / 'share' / 'valorant-ai-coaching')
)
DATASET_DIR = Path(os.environ.get('SPIKE_DATASET_DIR', Path.home() / 'datasets' / 'minimap_dot_spike'))
RAW_DIR = DATASET_DIR / 'raw'
YOLO_DIR = DATASET_DIR / 'yolo'
RAW_DIR.mkdir(parents=True, exist_ok=True)

# --- 録画/抽出パラメータ ---
RECORDING_W, RECORDING_H = 1280, 720
TARGET_FPS = 2          # 2fps で抽出
TARGET_CROP_COUNT = 200 # 集める総クロップ数
FFMPEG = os.environ.get('FFMPEG_PATH', 'ffmpeg')

# ミニマップ領域 (left, top, right, bottom) @ 1280x720。§0.5 で要キャリブレーション。
# VALORANT 既定ミニマップサイズの初期推定。実際の録画に合わせて必ず調整すること。
MINIMAP_BBOX = (0, 0, 300, 300)

# ラベルクラス (YOLO のクラス index 順)
CLASSES = ['local_player', 'ally_dot', 'enemy_dot', 'dead_marker']

videos = sorted(RECORDINGS_DIR.glob('*.mp4'))
print(f'Recordings dir : {RECORDINGS_DIR}  ({len(videos)} videos)')
print(f'Dataset dir    : {DATASET_DIR}')
print(f'Minimap bbox   : {MINIMAP_BBOX}')
for v in videos[:10]:
    print('  -', v.name)
if not videos:
    print('\n[!] 録画が見つかりません。RECORDINGS_DIR を確認するか、アプリで1試合録画してください。')

## 0.5 ミニマップ領域のキャリブレーション

最初の録画から 1 フレーム取り出し、現在の `MINIMAP_BBOX` を赤枠で重ねて表示する。枠がミニマップにぴったり重なるよう、§1 の `MINIMAP_BBOX` を調整して両セルを再実行する。一度合わせれば固定でよい。

In [ ]:
from PIL import Image, ImageDraw

def grab_frame(video_path, t_sec=30):
    """動画の t_sec 地点から1フレームを PIL Image で返す。"""
    with tempfile.TemporaryDirectory() as td:
        out = Path(td) / 'frame.jpg'
        subprocess.run(
            [FFMPEG, '-ss', str(t_sec), '-i', str(video_path), '-frames:v', '1', '-q:v', '2', '-y', str(out)],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        return Image.open(out).convert('RGB')

if videos:
    frame = grab_frame(videos[0], t_sec=30)
    print('frame size:', frame.size, '(expected ~1280x720)')
    preview = frame.copy()
    ImageDraw.Draw(preview).rectangle(MINIMAP_BBOX, outline=(255, 0, 0), width=3)
    crop = frame.crop(MINIMAP_BBOX)
    # Notebook 上で表示
    try:
        from IPython.display import display
        display(preview.resize((640, 360)))
        display(crop)
    except Exception:
        preview.save(DATASET_DIR / 'calib_full.png')
        crop.save(DATASET_DIR / 'calib_crop.png')
        print('saved calib_full.png / calib_crop.png to', DATASET_DIR)
else:
    print('録画が無いためスキップ')

## 2. ミニマップクロップ抽出

全録画から 2fps でフレームを取り、`MINIMAP_BBOX` でクロップして `raw/` に最大 `TARGET_CROP_COUNT` 枚保存する。複数試合に均等割り当てして偏りを避ける。

In [ ]:
def extract_frames(video_path, out_dir, fps):
    """video を fps でフレーム抽出 (videoAnalyzer.js と同じ ffmpeg パターン)。"""
    out_dir.mkdir(parents=True, exist_ok=True)
    pattern = str(out_dir / 'frame_%05d.jpg')
    subprocess.run(
        [FFMPEG, '-i', str(video_path), '-vf', f'fps={fps}', '-q:v', '3', '-y', pattern],
        check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    return sorted(out_dir.glob('*.jpg'))

def build_crops():
    if not videos:
        print('録画が無いため抽出をスキップ')
        return 0
    per_video = max(1, TARGET_CROP_COUNT // len(videos))
    saved = 0
    for vi, video in enumerate(videos):
        if saved >= TARGET_CROP_COUNT:
            break
        with tempfile.TemporaryDirectory() as td:
            frames = extract_frames(video, Path(td), TARGET_FPS)
            if not frames:
                continue
            # この動画から per_video 枚を等間隔サンプリング
            step = max(1, len(frames) // per_video)
            for fi in range(0, len(frames), step):
                if saved >= TARGET_CROP_COUNT:
                    break
                crop = Image.open(frames[fi]).convert('RGB').crop(MINIMAP_BBOX)
                crop.save(RAW_DIR / f'crop_{vi:02d}_{fi:05d}.png')
                saved += 1
        print(f'  {video.name}: total saved={saved}')
    return saved

n = build_crops()
print(f'\n保存クロップ数: {n}  →  {RAW_DIR}')

## 3. 手動ラベル

`raw/` の PNG を **CVAT / Roboflow / Label Studio** のいずれかにアップロードし、`CLASSES`（`local_player` / `ally_dot` / `enemy_dot` / `dead_marker`）で box ラベリングする。

- `local_player`: 中央の自分（白/シェブロン）
- `ally_dot`: 味方（緑系）
- `enemy_dot`: 敵（赤系、見えるときのみ）
- `dead_marker`: 死体マーカー

YOLO 形式でエクスポートし `yolo/{images,labels}/{train,val}/` に配置（train160/val40 目安）。下のセルでラベル分布を確認する。

In [ ]:
def label_distribution():
    label_dir = YOLO_DIR / 'labels'
    if not label_dir.exists():
        print(f'まだラベルがありません: {label_dir}')
        return
    counts = Counter()
    n_images = 0
    for txt in label_dir.rglob('*.txt'):
        n_images += 1
        for line in txt.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1
    print(f'ラベル済み画像: {n_images}')
    for idx, name in enumerate(CLASSES):
        print(f'  {name:14s}: {counts.get(idx, 0)} boxes')
    if counts.get(CLASSES.index('enemy_dot'), 0) < 50:
        print('\n[!] enemy_dot のサンプルが少ない。mAP が不安定になりやすいので追加ラベル推奨。')

label_distribution()

## 4. YOLOv8n の fine-tune

`data.yaml` を生成して学習。Colab GPU 推奨。

In [ ]:
import yaml

def write_data_yaml():
    cfg = {
        'path': str(YOLO_DIR),
        'train': 'images/train',
        'val': 'images/val',
        'names': {i: n for i, n in enumerate(CLASSES)},
    }
    p = YOLO_DIR / 'data.yaml'
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(yaml.safe_dump(cfg, allow_unicode=True))
    print('wrote', p)
    return p

data_yaml = write_data_yaml()

# 学習 (data.yaml と画像/ラベルが揃ってから実行)
model = None
if (YOLO_DIR / 'images' / 'train').exists():
    from ultralytics import YOLO
    model = YOLO('yolov8n.pt')
    model.train(data=str(data_yaml), epochs=100, imgsz=640, batch=16, name='minimap_dot_spike')
else:
    print('画像が未配置のため学習をスキップ。§3 のラベリング完了後に再実行する。')

## 5. mAP / Precision / Recall を測定

クラス別 mAP@50 を出し、`enemy_dot` を判定基準 (0.7) と照合する。

In [ ]:
ENEMY_MAP_THRESHOLD = 0.7

def evaluate(model):
    metrics = model.val()
    print('クラス別 mAP@50:')
    results = {}
    for ci, name in enumerate(CLASSES):
        # ultralytics: metrics.box.maps は mAP50-95 のクラス別配列。
        try:
            m = float(metrics.box.maps[ci])
        except Exception:
            m = float('nan')
        results[name] = m
        print(f'  {name:14s}: {m:.3f}')
    enemy = results.get('enemy_dot', float('nan'))
    print()
    if enemy >= ENEMY_MAP_THRESHOLD:
        print(f'[DECISION] enemy_dot mAP={enemy:.3f} >= {ENEMY_MAP_THRESHOLD} -> YOLO 本学習で Phase 4 を進める')
    else:
        print(f'[DECISION] enemy_dot mAP={enemy:.3f} < {ENEMY_MAP_THRESHOLD} -> §6 の HSV+Hungarian フォールバックを採用')
    return results

if model is not None and (YOLO_DIR / 'images' / 'val').exists():
    eval_results = evaluate(model)
else:
    print('学習済みモデルが無いため評価をスキップ。')

## 6. HSV 色バケット + ハンガリアン法フォールバック

YOLO がダメだった場合の代替。各ドットを HSV で `enemy`(赤) / `ally`(緑) に分け、フレーム間で「位置距離」を Hungarian assignment して連続軌跡を作る。個別 ally の再識別は諦め、味方は連続軌跡、敵は見えた瞬間の点として扱う。

下のセルは **録画なしの合成テスト** 込みで、アルゴリズム自体を今すぐ検証できる。

In [ ]:
import cv2
import numpy as np
from scipy.optimize import linear_sum_assignment

# HSV しきい値 (OpenCV: H 0-179, S/V 0-255)。実映像で要微調整。
HSV_RANGES = {
    'enemy': [((0, 90, 90), (10, 255, 255)), ((170, 90, 90), (179, 255, 255))],  # 赤は2帯域
    'ally':  [((40, 60, 60), (90, 255, 255))],                                    # 緑
}
MIN_BLOB_AREA = 6   # px^2。ノイズ除去

def detect_dots(crop_rgb):
    """RGB ndarray → [{'team','xy'}]. 色マスク→連結成分の重心。"""
    hsv = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2HSV)
    dots = []
    for team, ranges in HSV_RANGES.items():
        mask = np.zeros(hsv.shape[:2], np.uint8)
        for lo, hi in ranges:
            mask |= cv2.inRange(hsv, np.array(lo), np.array(hi))
        n, _, stats, centroids = cv2.connectedComponentsWithStats(mask, connectivity=8)
        for i in range(1, n):  # 0 は背景
            if stats[i, cv2.CC_STAT_AREA] >= MIN_BLOB_AREA:
                dots.append({'team': team, 'xy': tuple(centroids[i])})
    return dots

def track(frames_dots, max_dist=40):
    """フレームごとの dots 列 → 各 dot に track_id を付与。team 内で Hungarian。"""
    tracks = []          # 出力: フレームごとの [{'team','xy','track_id'}]
    next_id = 0
    active = {}          # team -> list of (track_id, last_xy)
    for dots in frames_dots:
        frame_out = []
        by_team = {}
        for d in dots:
            by_team.setdefault(d['team'], []).append(d)
        for team, tdots in by_team.items():
            prev = active.get(team, [])
            assigned = {}
            if prev and tdots:
                cost = np.zeros((len(prev), len(tdots)))
                for i, (_tid, pxy) in enumerate(prev):
                    for j, d in enumerate(tdots):
                        cost[i, j] = np.hypot(pxy[0] - d['xy'][0], pxy[1] - d['xy'][1])
                rows, cols = linear_sum_assignment(cost)
                for r, c in zip(rows, cols):
                    if cost[r, c] <= max_dist:
                        assigned[c] = prev[r][0]
            new_active = []
            for j, d in enumerate(tdots):
                tid = assigned.get(j)
                if tid is None:
                    tid = next_id; next_id += 1
                frame_out.append({**d, 'track_id': tid})
                new_active.append((tid, d['xy']))
            active[team] = new_active
        tracks.append(frame_out)
    return tracks

# --- 合成テスト: 緑2点が右に移動 + 赤1点が点滅。id が連続するか確認 ---
def _synthetic_frames():
    frames = []
    for t in range(5):
        img = np.zeros((100, 100, 3), np.uint8)
        cv2.circle(img, (10 + 4 * t, 20), 3, (0, 255, 0), -1)   # ally A →
        cv2.circle(img, (10 + 4 * t, 60), 3, (0, 255, 0), -1)   # ally B →
        if t % 2 == 0:
            cv2.circle(img, (80, 80), 3, (255, 0, 0), -1)         # enemy 点滅
        frames.append(img)
    return frames

syn = _synthetic_frames()
tracked = track([detect_dots(f) for f in syn])
ally_ids_per_frame = [sorted(d['track_id'] for d in fr if d['team'] == 'ally') for fr in tracked]
print('ally track ids / frame:', ally_ids_per_frame)
assert all(ids == ally_ids_per_frame[0] for ids in ally_ids_per_frame), 'ally の id が連続していない'
print('OK: 味方2点の track id がフレーム間で安定')
print('enemy detections / frame:', [sum(1 for d in fr if d['team'] == 'enemy') for fr in tracked])

## 7. 決定セクション

(本ノートブックを実行した後、ここに結果と Phase 4 への申し送りを記入する)

### データセット
- 抽出枚数: TBD
- ラベル分布: TBD

### YOLO 結果 (クラス別 mAP@50)
- local_player: TBD
- ally_dot    : TBD
- enemy_dot   : TBD
- dead_marker : TBD

### 採用する手法
- [ ] YOLO 本学習 (`enemy_dot` mAP ≥ 0.7 の場合)
- [ ] HSV + Hungarian フォールバック (上記未達の場合)

### Phase 4 (#13) への申し送り
- HSV しきい値の最終値 / MINIMAP_BBOX の最終値:
- track break (スモーク/オクルージョン) の頻度と対処:
- TBD